In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix, hstack
from scipy.sparse import save_npz

DATA_RAW = "../../data/raw/"
DATA_PROCESSED = "../../data/processed/"

In [19]:
reviews_first = pd.read_json(
    DATA_RAW + "Movies_and_TV.jsonl",
    lines=True,
    nrows=10  
)

print(reviews_first.iloc[2])

rating                                                               3
title                                     Some decent moments...but...
text                 Annabella Sciorra did her character justice wi...
images                                                              []
asin                                                        B096Z8Z3R6
parent_asin                                                 B096Z8Z3R6
user_id                                   AG2L7H23R5LLKDKLBEF2Q3L2MVDA
timestamp                                   2022-03-03 01:43:54.582000
helpful_vote                                                         0
verified_purchase                                                 True
Name: 2, dtype: object


In [20]:
meta = pd.read_json(
    DATA_RAW + "meta_Movies_and_TV.jsonl",
    lines=True,
    nrows=10
)

print(meta.iloc[3])

main_category                                            Prime Video
title                         Ode to Joy: Beethoven's Symphony No. 9
subtitle                                                         NaN
average_rating                                                   4.3
rating_number                                                     35
features                                     [1 h 26 min, 2015, ALL]
description        [This special Ode to Joy: Beethoven's Symphony...
price                                                           5.99
images             [{'360w': 'https://m.media-amazon.com/images/G...
videos                                                            []
store                                                            NaN
categories                                             [Documentary]
details            {'Audio languages': ['English'], 'Subtitles': ...
parent_asin                                               B01G9ILXXE
bought_together                   

### Processed Users (reviewerID → user_id (int))

In [21]:
user_map = {}
user_counts = {}
user_counter = 0

reviews = pd.read_json(
    DATA_RAW + "Movies_and_TV.jsonl",
    lines=True,
    chunksize=100_000
)

for chunk in reviews:
    
    # keep only needed column
    chunk = chunk[["user_id"]].dropna()
    
    for u in chunk["user_id"]:
        
        # assign integer id
        if u not in user_map:
            user_map[u] = user_counter
            user_counter += 1
        
        # count interactions
        user_counts[u] = user_counts.get(u, 0) + 1

users_df = pd.DataFrame({
    "original_user_id": list(user_map.keys()),
    "user_id": list(user_map.values())
})

users_df["num_reviews"] = users_df["original_user_id"].map(user_counts)

users_df.to_csv(DATA_PROCESSED + "users.csv", index=False)

### Processed movies

In [ ]:
import re
import pandas as pd

def extract_year_from_features(features):
    """
    Extract year ONLY from features list.
    """
    if not isinstance(features, list):
        return None

    for f in features:
        f = str(f).strip()

        # match year like 1998, 2013
        match = re.search(r"(19\d{2}|20[0-2]\d)", f)
        if match:
            return int(match.group(0))

    return None


item_map = {}
items_data = []
item_counter = 0

metas = pd.read_json(
    DATA_RAW + "meta_Movies_and_TV.jsonl",
    lines=True,
    chunksize=100_000
)

for chunk in metas:
    
    chunk = chunk[[
        "parent_asin", "title", "categories", "description", "features", "average_rating", "rating_number"
    ]].dropna(subset=["parent_asin"])
    
    for row in chunk.itertuples(index=False):
        
        p = row.parent_asin
        
        if p not in item_map:
            
            item_map[p] = item_counter
            
            # ---- Handle missing properly (store None) ----
            title = row.title if pd.notna(row.title) else None
            
            categories = (
                " | ".join(row.categories)
                if isinstance(row.categories, list)
                else None
            )
            
            description = (
                " ".join(row.description)
                if isinstance(row.description, list)
                else None
            )

            features = row.features if isinstance(row.features, list) else None
            
            # ---- Extract year ONLY from features ----
            year = extract_year_from_features(features)

            average_rating = row.average_rating if pd.notna(row.average_rating) else None
            rating_number = row.rating_number if pd.notna(row.rating_number) else None
            
            # ---- Extract year ONLY from features ----
            year = extract_year_from_features(features)

            if(title is None):
                continue
            
            # ---- Store result ----
            items_data.append({
                "item_id": item_counter,
                "parent_asin": p,
                "title": title,
                "categories": categories,
                "description": description,
                "year": year,
                "average_rating": average_rating,
                "rating_number": rating_number
            })
            
            item_counter += 1


# Convert to DataFrame
items = pd.DataFrame(items_data)
indexs = pd.DataFrame(item_map.items(), columns=["parent_asin", "item_id"])

# Save
items.to_csv(DATA_PROCESSED + "movies.csv", index=False)
indexs.to_csv(DATA_PROCESSED + "movie_indices.csv", index=False)

## Collabrative Filtering
### Processed ratings

In [25]:
import pandas as pd

ratings = pd.read_json(
    DATA_RAW + "Movies_and_TV.jsonl",
    lines=True,
    chunksize=100000
)

processed_chunks = []

for chunk in ratings:
    # Concatenate title and text for better content representation
    chunk["text"] = chunk["title"].fillna("") + ". " + chunk["text"].fillna("")
    
    # Keep only needed columns
    chunk = chunk[["user_id", "parent_asin", "text", "rating", "timestamp"]].dropna()
    
    # ---- Map IDs ----
    chunk["user_id"] = chunk["user_id"].map(user_map)
    chunk["item_id"] = chunk["parent_asin"].map(item_map)
    
    # Drop failed mappings
    chunk = chunk.dropna(subset=["user_id", "item_id"])
    
    # ---- Convert types ----
    chunk["user_id"] = chunk["user_id"].astype("int32")
    chunk["item_id"] = chunk["item_id"].astype("int32")
    chunk["rating"] = chunk["rating"].astype("float32")
    
    # Keep final columns
    chunk = chunk[["user_id", "item_id", "text", "rating", "timestamp"]]
    
    processed_chunks.append(chunk)

# Combine
ratings = pd.concat(processed_chunks, ignore_index=True)

# Keep only the latest rating for each user-item pair
ratings = ratings.sort_values("timestamp")
ratings = ratings.drop_duplicates(subset=["user_id", "item_id"], keep="last")

#Remove those users who have less than 10 ratings
user_rating_counts = ratings["user_id"].value_counts()
users_to_keep = user_rating_counts[user_rating_counts >= 10].index
ratings = ratings[ratings["user_id"].isin(users_to_keep)]

#Remove those items which have less than 10 ratings
item_rating_counts = ratings["item_id"].value_counts()
items_to_keep = item_rating_counts[item_rating_counts >= 10].index
ratings = ratings[ratings["item_id"].isin(items_to_keep)]

# Save
ratings.to_csv(DATA_PROCESSED + "ratings_clean.csv", index=False)

In [26]:
# Remove movies from movies.csv that is not in ratings_clean.csv
movies = items[items["item_id"].isin(ratings["item_id"].unique())]
movies.to_csv(DATA_PROCESSED + "movies.csv", index=False)


### User-Item Matrix

In [27]:
ratings = pd.read_csv(DATA_PROCESSED + "ratings_clean.csv", parse_dates=["timestamp"])

# ---- Remove duplicates (keep latest rating) ----
ratings = ratings.sort_values("timestamp")
ratings = ratings.drop_duplicates(
    subset=["user_id", "item_id"],
    keep="last"
)

# ---- Stable indexing ----
user_ids = np.sort(ratings["user_id"].unique())
item_ids = np.sort(ratings["item_id"].unique())

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {i: j for j, i in enumerate(item_ids)}

# ---- Map indices ----
ratings["user_idx"] = ratings["user_id"].map(user2idx)
ratings["item_idx"] = ratings["item_id"].map(item2idx)

# ---- Build sparse matrix ----
user_item_matrix = csr_matrix(
    (
        ratings["rating"].astype(np.float32),
        (ratings["user_idx"], ratings["item_idx"])
    ),
    shape=(len(user_ids), len(item_ids))
)

# ---- Save ----
save_npz(DATA_PROCESSED + "user_item_matrix.npz", user_item_matrix)

## Content Based Filtering
### Item Features Matrix

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

movies_meta = pd.read_csv(DATA_PROCESSED + "movies.csv")
movies_meta = movies_meta.sort_values("item_id").reset_index(drop=True)
title_text = movies_meta["title"].fillna("").astype(str)

categories_text = (
    movies_meta["categories"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s*\|\s*", " ", regex=True)
)

description_text = movies_meta["description"].fillna("").astype(str)

# ---- Build corpus (include title) ----
corpus = (
    title_text + " " +
    categories_text + " " +
    description_text
).str.strip()

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_features=100_000,
    ngram_range=(1, 2)
)

item_features = tfidf.fit_transform(corpus)

save_npz(DATA_PROCESSED + "item_features.npz", item_features)

In [29]:
index_df = pd.DataFrame({
    "row_idx": np.arange(len(movies_meta)),
    "item_id": movies_meta["item_id"].values
})

index_df.to_csv(DATA_PROCESSED + "item_features_index.csv", index=False)

# Need this because when a new item comes in at runtime, 
# you need to transform its text using the same vocabulary 
# and weights that were fitted on your training data. If you 
# don't save it, you'd have to retrain from scratch every time.
import joblib
joblib.dump(tfidf, DATA_PROCESSED + "tfidf_vectorizer.pkl")

['../../data/processed/tfidf_vectorizer.pkl']